## Import Libraries

In [1]:
from nilearn.image import clean_img, mean_img
import nibabel as nib
import numpy as np
import pandas as pd
from bids import BIDSLayout
import os

## Define BIDS Layout

In [2]:
# Setup Layout and Groups
data_path = '/Volumes/T9/ds001486/derivatives/fmriprep'
layout = BIDSLayout(data_path, validate=False, derivatives=False)
bold_files = layout.get(suffix='bold', extension='nii.gz', desc='preproc', return_type='file')

# Defined subject groups
mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
td_subs = ['090', '036', '013', '008', '057', '070', '023', '024', '053', '044', '034', '060', '007', '027', '010']

output_dir = os.path.expanduser('~/Desktop/FINAL_brainmath_cleaned_nifti')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for bold_path in bold_files:
    filename = os.path.basename(bold_path)
    
    # Task Filtering (Only Math tasks)
    if 'task-Mult' in filename:
        current_task = 1
    elif 'task-Sub' in filename:
        current_task = 0
    else:
        continue 

    # Confound Handling
    parts = filename.split('_')

    sub_part = [p for p in parts if p.startswith('sub-')]
    sub_id = sub_part[0].split('-')[1] if sub_part else "unknown"
    print(f"Processing {filename} for subject {sub_id}...")

    # 3. Extract Session ID (Handles 'T2', '01', etc.)
    ses_part = [p for p in parts if p.startswith('ses-')]
    ses_id = ses_part[0].split('-')[1] if ses_part else "None"
    print(f"Session ID: {ses_id}")

    # 4. Extract Run ID (If it exists)
    run_part = [p for p in parts if p.startswith('run-')]
    run_id = run_part[0].split('-')[1] if run_part else "01"
    print(f"Run ID: {run_id}")

    essential_parts = [p for p in parts if any(x in p for x in ['sub-', 'ses-', 'task-', 'run-'])]
    confound_name = "_".join(essential_parts) + "_desc-confounds_timeseries.tsv"
    confound_path = os.path.join(os.path.dirname(bold_path), confound_name)

    if not os.path.exists(confound_path):
        print(f"FAILURE: Confound file not found for {filename}, skipping.")
        continue

    print(f"SUCCESS: Confound found for {filename}, Processing...")
    sub_id = filename.split('_')[0].split('-')[1] 
    group_label = 1 if sub_id in mld_subs else 0 

    try:
        # 1. Load and clean your confounds (Exactly like your old code)
        df = pd.read_csv(confound_path, sep='\t')
        df_clean = df.select_dtypes(include=[np.number]).fillna(0).dropna(axis=1, how='all')
        
        # 2. THE CHANGE: Clean the 4D image directly, keeping spatial dimensions intact
        # This replaces masker.fit_transform()
        cleaned_4d_img = clean_img(bold_path, confounds=df_clean, standardize=None)
        
        # 3. Compress the 4th dimension (Time) into a 3D summary volume
        # Taking the mean or variance across the time axis
        final_3d_img = mean_img(cleaned_4d_img) 
        
        # 4. Save the new 3D NIfTI file
        unique_id = f"sub-{sub_id}_ses-{ses_id}_task-{'Mult' if current_task==1 else 'Sub'}_run-{run_id}"
        save_path = os.path.join(output_dir, f'{unique_id}_cleaned3D.nii.gz')
        
        nib.save(final_3d_img, save_path)
        print(f"Saved: {save_path}")
        
    except Exception as e:
        print(f"Error on sub-{sub_id}: {e}")

Processing sub-007_ses-T1_task-Mult_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz for subject 007...
Session ID: T1
Run ID: 01
SUCCESS: Confound found for sub-007_ses-T1_task-Mult_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz, Processing...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_35284/2295912916.py:61: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_4d_img = clean_img(bold_path, confounds=df_clean, standardize=None)


Saved: /Users/jchong058/Desktop/FINAL_brainmath_cleaned_nifti/sub-007_ses-T1_task-Mult_run-01_cleaned3D.nii.gz
Processing sub-007_ses-T1_task-Mult_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz for subject 007...
Session ID: T1
Run ID: 02
SUCCESS: Confound found for sub-007_ses-T1_task-Mult_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz, Processing...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_35284/2295912916.py:61: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_4d_img = clean_img(bold_path, confounds=df_clean, standardize=None)


Saved: /Users/jchong058/Desktop/FINAL_brainmath_cleaned_nifti/sub-007_ses-T1_task-Mult_run-02_cleaned3D.nii.gz
Processing sub-007_ses-T1_task-Sub_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz for subject 007...
Session ID: T1
Run ID: 01
SUCCESS: Confound found for sub-007_ses-T1_task-Sub_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz, Processing...


...

Saved: /Users/jchong058/Desktop/FINAL_brainmath_cleaned_nifti/sub-106_ses-T2_task-Mult_run-02_cleaned3D.nii.gz
Processing sub-106_ses-T2_task-Sub_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz for subject 106...
Session ID: T2
Run ID: 01
SUCCESS: Confound found for sub-106_ses-T2_task-Sub_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz, Processing...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_35284/2295912916.py:61: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_4d_img = clean_img(bold_path, confounds=df_clean, standardize=None)


Saved: /Users/jchong058/Desktop/FINAL_brainmath_cleaned_nifti/sub-106_ses-T2_task-Sub_run-01_cleaned3D.nii.gz
Processing sub-106_ses-T2_task-Sub_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz for subject 106...
Session ID: T2
Run ID: 02
SUCCESS: Confound found for sub-106_ses-T2_task-Sub_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz, Processing...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_35284/2295912916.py:61: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_4d_img = clean_img(bold_path, confounds=df_clean, standardize=None)


Saved: /Users/jchong058/Desktop/FINAL_brainmath_cleaned_nifti/sub-106_ses-T2_task-Sub_run-02_cleaned3D.nii.gz
